# Parkinson's Disease Prediction using fNIRS Signals
## A Comparative Study: Classical ML vs Variational Quantum Classifier (VQC)

**Dataset**: Zenodo — `fNIRS_Parkinson.zip`  
**Tasks**: `finger_tapping` | `resting_state` | `walking`  
**Format**: `.snirf` (HDF5-based standard fNIRS format)  
**Subjects**: 20 PD + 20 Control  
**Task**: Binary Classification — Parkinson's Disease vs Healthy Controls

---
 


## 0. Install Dependencies

In [20]:
!pip install pennylane pennylane-lightning mne h5py scipy scikit-learn pandas numpy matplotlib seaborn snirf -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [21]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import h5py
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.decomposition import PCA

import pennylane as qml
from pennylane import numpy as pnp

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

print('Imports OK')
print(f'PennyLane: {qml.__version__}')

Imports OK
PennyLane: 0.44.1


## 2. Configuration

> **Edit `BASE_DIR`** to point to your extracted `fNIRS_Parkinson` folder.  
> Choose which task to classify.

In [22]:
from pathlib import Path

# ─────────────────────────────────────────
BASE_DIR = Path('./fNIRS_Parkinson')
TASK = 'finger_tapping'
# ─────────────────────────────────────────

TASK_FILE = {
    'finger_tapping': '2_finger_tapping.snirf',
    'resting_state':  '1_resting_seg_1.snirf',
    'walking':        '3_walking.snirf'
}

TASK_DIR   = BASE_DIR / TASK
SNIRF_FILE = TASK_FILE[TASK]

print(f'Task      : {TASK}')
print(f'Task dir  : {TASK_DIR}')
print(f'Dir exists: {TASK_DIR.exists()}')

# ─────────────────────────────────────────
# Collect all SNIRF files dynamically
# ─────────────────────────────────────────

data = []

for group in ['PD', 'Control']:   # adjust if your control folder name differs
    group_dir = TASK_DIR / group
    
    if not group_dir.exists():
        print(f"Missing group folder: {group_dir}")
        continue
    
    for subject in group_dir.iterdir():
        if subject.is_dir():
            snirf_path = subject / SNIRF_FILE
            
            if snirf_path.exists():
                data.append({
                    'group': group,
                    'subject': subject.name,
                    'path': snirf_path
                })
            else:
                print(f"Missing file in {subject}")

# ─────────────────────────────────────────
# Summary
# ─────────────────────────────────────────

print(f"\nTotal files found: {len(data)}")

# Preview
for d in data[:5]:
    print(d)

Task      : finger_tapping
Task dir  : fNIRS_Parkinson\finger_tapping
Dir exists: True

Total files found: 40
{'group': 'PD', 'subject': 'PD01', 'path': WindowsPath('fNIRS_Parkinson/finger_tapping/PD/PD01/2_finger_tapping.snirf')}
{'group': 'PD', 'subject': 'PD02', 'path': WindowsPath('fNIRS_Parkinson/finger_tapping/PD/PD02/2_finger_tapping.snirf')}
{'group': 'PD', 'subject': 'PD03', 'path': WindowsPath('fNIRS_Parkinson/finger_tapping/PD/PD03/2_finger_tapping.snirf')}
{'group': 'PD', 'subject': 'PD04', 'path': WindowsPath('fNIRS_Parkinson/finger_tapping/PD/PD04/2_finger_tapping.snirf')}
{'group': 'PD', 'subject': 'PD05', 'path': WindowsPath('fNIRS_Parkinson/finger_tapping/PD/PD05/2_finger_tapping.snirf')}


## 3. SNIRF Data Loader

SNIRF is HDF5. We read it directly with `h5py`. No MATLAB needed.

In [23]:
def read_snirf(filepath):
    """
    Read one .snirf file.
    Returns:
      raw      : np.array (n_timepoints, n_channels)
      times    : np.array (n_timepoints,)
      fs       : float  sampling frequency (Hz)
      ch_types : list of str  e.g. ['hbo','hbo',...,'hbr','hbr',...]
    """
    with h5py.File(filepath, 'r') as f:
        nirs = f['nirs']
        # Data group can be 'data1' or 'data'
        data_key = 'data1' if 'data1' in nirs else 'data'
        dgrp = nirs[data_key]

        raw   = dgrp['dataTimeSeries'][()]   # (n_tp, n_ch)
        times = dgrp['time'][()]             # (n_tp,)

        fs = 1.0 / np.median(np.diff(times)) if len(times) > 1 else 10.0

        # Parse channel types from measurementList
        ch_types = []
        i = 1
        while f'measurementList{i}' in dgrp:
            ml = dgrp[f'measurementList{i}']
            dt = int(ml['dataType'][()]) if 'dataType' in ml else -1
            if   dt == 99001: ch_types.append('hbo')
            elif dt == 99002: ch_types.append('hbr')
            else:
                wl = int(ml['wavelengthIndex'][()]) if 'wavelengthIndex' in ml else i
                ch_types.append(f'raw_wl{wl}')
            i += 1

    return raw, times, fs, ch_types


def load_dataset(task_dir, snirf_file):
    """
    Load all subjects for a task.
    Structure: task_dir / {Control|PD} / {SubjectID} / snirf_file
    Returns:
      X_list   : list of (n_tp, n_ch) arrays  (variable length)
      y        : np.array labels (0=HC, 1=PD)
      fs_list  : list of sampling frequencies
      names    : list of subject ID strings
    """
    X_list, y, fs_list, names = [], [], [], []

    for group, label in [('Control', 0), ('PD', 1)]:
        group_dir = task_dir / group
        if not group_dir.exists():
            print(f'WARNING: {group_dir} not found')
            continue
        for subj_dir in sorted(group_dir.iterdir()):
            fpath = subj_dir / snirf_file
            if not fpath.exists():
                print(f'  Missing: {fpath}')
                continue
            try:
                raw, times, fs, ch_types = read_snirf(fpath)
                X_list.append(raw)
                y.append(label)
                fs_list.append(fs)
                names.append(f'{group}/{subj_dir.name}')
            except Exception as e:
                print(f'  Error {fpath.name}: {e}')

    return X_list, np.array(y), fs_list, names


print('Loading data...')
X_raw_list, y, fs_list, subject_names = load_dataset(TASK_DIR, SNIRF_FILE)

FS = float(np.median(fs_list))
N_SUBJECTS = len(X_raw_list)

print(f'\nDataset Summary  [{TASK}]')
print(f'  Subjects   : {N_SUBJECTS}')
print(f'  PD / HC    : {(y==1).sum()} / {(y==0).sum()}')
print(f'  Fs         : {FS:.2f} Hz')
print(f'  Channels   : {X_raw_list[0].shape[1]}')
print(f'  Timepoints : {[x.shape[0] for x in X_raw_list[:5]]} (first 5 subjects)')

Loading data...

Dataset Summary  [finger_tapping]
  Subjects   : 40
  PD / HC    : 20 / 20
  Fs         : 25.00 Hz
  Channels   : 44
  Timepoints : [32821, 16301, 59953, 16563, 16846] (first 5 subjects)


In [24]:
import mne

def convert_to_hb(filepath):
    raw = mne.io.read_raw_snirf(filepath, preload=True, verbose=False)
    
    # Convert to Optical Density
    raw_od = mne.preprocessing.nirs.optical_density(raw)
    
    # Convert to HbO / HbR
    raw_hb = mne.preprocessing.nirs.beer_lambert_law(raw_od)
    
    data = raw_hb.get_data().T   # (time, channels)
    ch_types = raw_hb.get_channel_types()
    
    return data, ch_types


# --- Apply on first subject to get indices ---
first_group = 'Control'
first_subj  = sorted((TASK_DIR / first_group).iterdir())[0]
fpath = first_subj / SNIRF_FILE

X_sample, ch_types_all = convert_to_hb(fpath)

# Identify indices
hbo_idx = [i for i, t in enumerate(ch_types_all) if t == 'hbo']
hbr_idx = [i for i, t in enumerate(ch_types_all) if t == 'hbr']

print(f'HbO channels: {len(hbo_idx)}')
print(f'HbR channels: {len(hbr_idx)}')
print(f'Sample ch types (first 8): {ch_types_all[:8]}')



HbO channels: 22
HbR channels: 22
Sample ch types (first 8): ['hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo']


In [25]:
X_list, y, names = [], [], []

for group, label in [('Control', 0), ('PD', 1)]:
    group_dir = TASK_DIR / group
    
    for subj_dir in sorted(group_dir.iterdir()):
        fpath = subj_dir / SNIRF_FILE
        
        if not fpath.exists():
            continue
        
        try:
            X, ch_types = convert_to_hb(fpath)
            X_list.append(X)
            y.append(label)
            names.append(f'{group}/{subj_dir.name}')
        except Exception as e:
            print(f'Error: {fpath}', e)

y = np.array(y)
print(X_list,y)

[array([[ 1.46579487e-03,  2.06515264e-03,  5.18432129e-03, ...,
         7.14208016e-04,  5.79313807e-04,  8.24423044e-04],
       [ 1.46579487e-03,  2.06515264e-03,  5.18432129e-03, ...,
         7.14208016e-04,  5.79313807e-04,  8.24423044e-04],
       [-1.11854790e-05,  2.51710342e-06, -9.92071415e-07, ...,
         2.72057058e-06, -2.43088792e-05, -2.04081479e-08],
       ...,
       [ 6.64658623e-06,  3.35249001e-06,  4.75743721e-06, ...,
        -1.99329098e-06,  6.30604888e-07, -1.99555671e-06],
       [ 6.42494467e-06,  3.08200452e-06,  4.27016017e-06, ...,
        -2.04152029e-06,  6.30604888e-07, -2.01000528e-06],
       [ 6.20356545e-06,  2.82755068e-06,  3.81092310e-06, ...,
        -2.04094609e-06,  6.99257752e-07, -2.02469925e-06]],
      shape=(32821, 44)), array([[ 1.50160486e-03,  1.81903485e-03,  5.67963195e-03, ...,
         6.75032511e-04,  3.94518061e-04,  4.83162860e-04],
       [ 1.50160486e-03,  1.81903485e-03,  5.67963195e-03, ...,
         6.75032511e-04,  3.

## 4. Preprocessing
- Trim to shortest subject
- Bandpass filter 0.01–0.1 Hz
- Z-score normalization per channel

In [26]:
def bandpass_filter(data, fs, lowcut=0.01, highcut=0.1, order=3):
    """data: (n_tp, n_ch)"""
    nyq  = 0.5 * fs
    b, a = scipy_signal.butter(order, [lowcut/nyq, min(highcut/nyq, 0.99)], btype='band')
    return scipy_signal.filtfilt(b, a, data, axis=0)

def zscore(data):
    """data: (n_tp, n_ch)"""
    return (data - data.mean(0)) / (data.std(0) + 1e-8)

min_tp = min(x.shape[0] for x in X_raw_list)
print(f'Min timepoints across subjects: {min_tp}  ({min_tp/FS:.1f} s)')
print(f'Max timepoints: {max(x.shape[0] for x in X_raw_list)}')
print(f'Trimming all to {min_tp} timepoints...')

X_proc_list = []
for sig in X_list:
    s = sig[:min_tp, :]          # trim
    s = bandpass_filter(s, FS)   # bandpass
    s = zscore(s)                # normalize
    X_proc_list.append(s)

# Stack → (n_subj, n_tp, n_ch) → transpose → (n_subj, n_ch, n_tp)
X_proc = np.stack(X_proc_list).transpose(0, 2, 1)
N_SUBJECTS, N_CHANNELS, N_TIMEPOINTS = X_proc.shape
print(f'\nPreprocessed shape: {X_proc.shape}  [subjects x channels x timepoints]')


# Visualization


Min timepoints across subjects: 8224  (329.0 s)
Max timepoints: 59953
Trimming all to 8224 timepoints...

Preprocessed shape: (40, 44, 8224)  [subjects x channels x timepoints]


## 5. Feature Extraction

In [37]:
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew, kurtosis, ttest_ind

FEAT_NAMES = ['mean','std','skew','kurt','min','max','range','mad','energy','slope']

def channel_features(sig):
    return [
        np.mean(sig),
        np.std(sig),
        float(skew(sig)),
        float(kurtosis(sig)),
        np.min(sig),
        np.max(sig),
        np.ptp(sig),
        np.mean(np.abs(sig)),
        float(np.mean(sig**2)),   # ✅ FIXED (normalized energy)
        float(np.polyfit(np.arange(len(sig)), sig, 1)[0])
    ]

def extract_features(X):
    """X: (n_subj, n_ch, n_tp)"""
    return np.array([
        [f for ch in range(X.shape[1]) for f in channel_features(X[i, ch, :])]
        for i in range(X.shape[0])
    ])

print('Extracting features...')
X_feat = extract_features(X_proc)

# ✅ Feature scaling (CRUCIAL)
scaler = StandardScaler()
X_feat = scaler.fit_transform(X_feat)

print(f'Feature matrix: {X_feat.shape}  ({N_CHANNELS} ch x 10 feats = {X_feat.shape[1]})')


Extracting features...
Feature matrix: (40, 440)  (44 ch x 10 feats = 440)


In [59]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_feat, y, test_size=0.2, random_state=42, stratify=y
)
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train_s.shape}  |  Test: {X_test_s.shape}')
print(f'Train PD/HC: {(y_train==1).sum()}/{(y_train==0).sum()}')
print(f'Test  PD/HC: {(y_test==1).sum()}/{(y_test==0).sum()}')

# PCA for VQC
N_QUBITS   = 6
pca        = PCA(n_components=N_QUBITS, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
print(f'\nPCA explained variance ({N_QUBITS} PCs): {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Scale to [-pi, pi] for angle encoding
pca_sc    = StandardScaler()
X_train_q = np.clip(pca_sc.fit_transform(X_train_pca) * np.pi/3, -np.pi, np.pi)
X_test_q  = np.clip(pca_sc.transform(X_test_pca)  * np.pi/3, -np.pi, np.pi)

Train: (32, 440)  |  Test: (8, 440)
Train PD/HC: 16/16
Test  PD/HC: 4/4

PCA explained variance (6 PCs): 63.4%


## 6. Classical ML Models

In [53]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    balanced_accuracy_score
)
import numpy as np

def evaluate(clf, Xtr, ytr, Xte, yte, name, cv=5):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    # Cross-validation (Accuracy + AUC)
    cv_acc = cross_val_score(clf, Xtr, ytr, cv=skf, scoring='accuracy')
    
    try:
        cv_auc = cross_val_score(clf, Xtr, ytr, cv=skf, scoring='roc_auc')
        cv_auc_mean = cv_auc.mean()
    except:
        cv_auc_mean = float('nan')
    
    # Train
    clf.fit(Xtr, ytr)
    
    # Predictions
    y_pred = clf.predict(Xte)
    
    # Probabilities / decision scores
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(Xte)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_prob = clf.decision_function(Xte)
    else:
        y_prob = None

    # Metrics
    acc  = accuracy_score(yte, y_pred)
    f1   = f1_score(yte, y_pred, zero_division=0)
    bal  = balanced_accuracy_score(yte, y_pred)
    
    # ✅ Robust AUC (fixes your issue)
    if y_prob is not None and len(np.unique(y_prob)) > 1:
        roc = roc_auc_score(yte, y_prob)
    else:
        roc = float('nan')
    
    cm = confusion_matrix(yte, y_pred)

    # Print results
    print(f'\n{name}')
    print(f'  CV Acc : {cv_acc.mean():.3f} +/- {cv_acc.std():.3f}')
    print(f'  CV AUC : {cv_auc_mean:.3f}')
    print(f'  Test   : Acc={acc:.3f}  F1={f1:.3f}  BalAcc={bal:.3f}  AUC={roc:.3f}')
    
    print('  Confusion Matrix:')
    print(cm)
    
    print('  Classification Report:')
    print(classification_report(yte, y_pred, zero_division=0))

    return {
        'name': name,
        'clf': clf,
        'cv_acc_mean': cv_acc.mean(),
        'cv_acc_std': cv_acc.std(),
        'cv_auc': cv_auc_mean,
        'test_acc': acc,
        'f1': f1,
        'bal_acc': bal,
        'auc': roc,
        'cm': cm,
        'y_pred': y_pred,
        'y_prob': y_prob
    }


# Models
classifiers = [
    #(LogisticRegression(C=1.0, max_iter=1000, random_state=42),                 'Logistic Regression'),
    #(RandomForestClassifier(n_estimators=200, random_state=42),                  'Random Forest'),
    (KNeighborsClassifier(n_neighbors=5),                                        'KNN'),
]

# Run evaluation
classical_results = [
    evaluate(clf, X_train_s, y_train, X_test_s, y_test, nm)
    for clf, nm in classifiers
]

print('\nClassical models complete.')


KNN
  CV Acc : 0.510 +/- 0.146
  CV AUC : 0.567
  Test   : Acc=0.750  F1=0.667  BalAcc=0.750  AUC=0.750
  Confusion Matrix:
[[4 0]
 [2 2]]
  Classification Report:
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       1.00      0.50      0.67         4

    accuracy                           0.75         8
   macro avg       0.83      0.75      0.73         8
weighted avg       0.83      0.75      0.73         8


Classical models complete.


## 7. Variational Quantum Classifier (VQC)

Architecture: `AngleEmbedding` → `StronglyEntanglingLayers` → measure `⟨Z₀⟩`  
Backend: `default.qubit` (CPU simulator)

In [60]:
N_LAYERS = 3
dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev)
def vqc(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))

W_SHAPE = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
print(f'Trainable params: {np.prod(W_SHAPE)}')
#print(qml.draw(vqc)(pnp.zeros(N_QUBITS), pnp.zeros(W_SHAPE)))
print(
    qml.draw(vqc, level="device")(
        pnp.zeros(N_QUBITS),        # inputs
        pnp.zeros(W_SHAPE)          # weights
    )
)

Trainable params: 54
0: ──RY(0.00)──Rot(0.00,0.00,0.00)─╭●─────────────╭X──Rot(0.00,0.00,0.00)─╭●──────────╭X ···
1: ──RY(0.00)──Rot(0.00,0.00,0.00)─╰X─╭●──────────│───Rot(0.00,0.00,0.00)─│──╭●───────│─ ···
2: ──RY(0.00)──Rot(0.00,0.00,0.00)────╰X─╭●───────│───Rot(0.00,0.00,0.00)─╰X─│──╭●────│─ ···
3: ──RY(0.00)──Rot(0.00,0.00,0.00)───────╰X─╭●────│───Rot(0.00,0.00,0.00)────╰X─│──╭●─│─ ···
4: ──RY(0.00)──Rot(0.00,0.00,0.00)──────────╰X─╭●─│───Rot(0.00,0.00,0.00)───────╰X─│──╰● ···
5: ──RY(0.00)──Rot(0.00,0.00,0.00)─────────────╰X─╰●──Rot(0.00,0.00,0.00)──────────╰X─── ···

0: ··· ──Rot(0.00,0.00,0.00)──────────────────────╭●───────╭X───────┤  <Z>
1: ··· ─╭X────────────────────Rot(0.00,0.00,0.00)─│──╭●────│──╭X────┤     
2: ··· ─│─────────────────────Rot(0.00,0.00,0.00)─│──│──╭●─│──│──╭X─┤     
3: ··· ─│─────────────────────Rot(0.00,0.00,0.00)─╰X─│──│──╰●─│──│──┤     
4: ··· ─│─────────────────────Rot(0.00,0.00,0.00)────╰X─│─────╰●─│──┤     
5: ··· ─╰●────────────────────Rot(0.00,0.00,0

In [61]:
def cost(weights, Xb, yb):
    raw   = pnp.array([vqc(x, weights) for x in Xb])
    prob  = pnp.clip((raw + 1) / 2, 1e-7, 1-1e-7)
    yb_   = pnp.array(yb, dtype=float)
    return -pnp.mean(yb_ * pnp.log(prob) + (1-yb_) * pnp.log(1-prob))


def q_predict(X, weights):
    raw  = np.array([float(vqc(x, weights)) for x in X])
    prob = (raw + 1) / 2
    return (prob > 0.5).astype(int), prob


# ─────────────────────────────────────────
# Initialization
# ─────────────────────────────────────────
rng = np.random.default_rng(42)
W   = pnp.array(rng.uniform(-np.pi, np.pi, W_SHAPE), requires_grad=True)

# 🔥 Slightly lower LR for stability
opt = qml.AdamOptimizer(stepsize=0.02)

N_EPOCHS  = 40
BATCH_SIZE = 8
n_tr = len(X_train_q)

losses, tr_accs, val_accs = [], [], []

best_acc   = 0
best_W     = None
best_epoch = 0

# 🔥 Early stopping settings
patience = 15

print(f'Training VQC: {N_EPOCHS} epochs | {N_QUBITS} qubits | {N_LAYERS} layers')
print('-' * 50)


# ─────────────────────────────────────────
# Training loop
# ─────────────────────────────────────────
for ep in range(N_EPOCHS):

    idx = rng.permutation(n_tr)
    ep_loss, n_b = 0, 0

    # Mini-batch training
    for s in range(0, n_tr, BATCH_SIZE):
        bi = idx[s:s+BATCH_SIZE]
        W, lv = opt.step_and_cost(lambda w: cost(w, X_train_q[bi], y_train[bi]), W)
        ep_loss += float(lv)
        n_b += 1

    # Evaluate
    tp, _ = q_predict(X_train_q, W)
    vp, _ = q_predict(X_test_q,  W)

    ta = accuracy_score(y_train, tp)
    va = accuracy_score(y_test,  vp)

    losses.append(ep_loss / n_b)
    tr_accs.append(ta)
    val_accs.append(va)

    # Save best model
    if va > best_acc:
        best_acc   = va
        best_W     = W.copy()
        best_epoch = ep

    # Logging
    if (ep+1) % 5 == 0:
        print(f'Ep {ep+1:3d}/{N_EPOCHS}  Loss:{losses[-1]:.4f}  Train:{ta:.3f}  Val:{va:.3f}')

    # 🔥 Early stopping
    if ep - best_epoch > patience:
        print(f'\nEarly stopping at epoch {ep+1}')
        break


# ─────────────────────────────────────────
# Final results
# ─────────────────────────────────────────
print(f'\nBest val acc: {best_acc:.3f} at epoch {best_epoch+1}')

# Evaluate best model
y_pred, y_prob = q_predict(X_test_q, best_W)
final_acc = accuracy_score(y_test, y_pred)

print(f'Final Test Accuracy (best model): {final_acc:.3f}')
print(W)

Training VQC: 40 epochs | 6 qubits | 3 layers
--------------------------------------------------
Ep   5/40  Loss:0.6823  Train:0.594  Val:0.750
Ep  10/40  Loss:0.6558  Train:0.688  Val:0.500
Ep  15/40  Loss:0.6313  Train:0.719  Val:0.750
Ep  20/40  Loss:0.6233  Train:0.750  Val:0.750

Early stopping at epoch 20

Best val acc: 0.750 at epoch 4
Final Test Accuracy (best model): 0.750
[[[ 1.69864452 -1.39643557  2.02385619]
  [ 1.37201086 -2.14885456  2.53838705]
  [ 2.03415957  1.36496413 -2.74720567]
  [-0.17384385 -0.4639306   2.18657593]
  [ 0.7384829   2.75931423 -1.36135231]
  [-1.92272385 -0.40027091 -2.75810498]]

 [[ 2.05856737  0.82727182  1.6216131 ]
  [-0.5075594   2.60917986  2.47005285]
  [ 1.7491351  -1.91864158 -0.2090981 ]
  [-2.52331271 -1.90127453  1.60140936]
  [ 1.53788598  2.93745028 -1.09437155]
  [-0.22436373 -0.09534581 -1.43679793]]

 [[-2.32527176 -0.15265045 -1.71587917]
  [ 1.0669728  -0.39488614  2.09027875]
  [ 1.25830275 -1.17893516  2.0876499 ]
  [ 1.51274

In [62]:
y_pred_vqc, y_prob_vqc = q_predict(X_test_q, best_W)
vqc_result = {
    'name': 'VQC (Quantum)',
    'cv_mean': float(np.mean(val_accs[-10:])),
    'cv_std':  float(np.std(val_accs[-10:])),
    'test_acc': accuracy_score(y_test, y_pred_vqc),
    'f1':       f1_score(y_test, y_pred_vqc, zero_division=0),
    'auc':      roc_auc_score(y_test, y_prob_vqc),
    'cm':       confusion_matrix(y_test, y_pred_vqc),
    'y_pred':   y_pred_vqc,
    'y_prob':   y_prob_vqc
}
print(classification_report(y_test, y_pred_vqc, target_names=['Control','PD']))

              precision    recall  f1-score   support

     Control       0.67      1.00      0.80         4
          PD       1.00      0.50      0.67         4

    accuracy                           0.75         8
   macro avg       0.83      0.75      0.73         8
weighted avg       0.83      0.75      0.73         8



## 8. Results & Visualizations

In [63]:
all_results = classical_results + [vqc_result]

results_df = pd.DataFrame([{
    'Model': r['name'],

    # ✅ Handle classical vs VQC naming difference
    'CV Acc': round(r.get('cv_acc_mean', r.get('cv_mean', np.nan)), 4),
    'CV Std': round(r.get('cv_acc_std',  r.get('cv_std',  np.nan)), 4),

    'Test Accuracy': round(r['test_acc'], 4),
    'F1 Score':      round(r['f1'], 4),
    'AUC-ROC':       round(r['auc'], 4)

} for r in all_results]).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)


print('=' * 70)
print(f'FINAL RESULTS — Task: {TASK}')
print('=' * 70)
print(results_df.to_string(index=False))
print('=' * 70)

FINAL RESULTS — Task: finger_tapping
        Model  CV Acc  CV Std  Test Accuracy  F1 Score  AUC-ROC
          KNN  0.5095  0.1457           0.75    0.6667   0.7500
VQC (Quantum)  0.7000  0.1000           0.75    0.6667   0.5625


# 10. Running on IBM Quantum Computer

In [64]:
# ============================================
# RUN VQC ON REAL IBM QUANTUM HARDWARE
# (LEAST BUSY VERSION - SAFE)
# ============================================

import pennylane as qml
from pennylane import numpy as pnp
import numpy as np

from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="bELiLIsO0qLEmU2YbRMDImddO8Mtlu2q5iI1-Rm_jHs4",
    overwrite=True
)

service = QiskitRuntimeService()

# ============================================
# Step 2: Find least busy backend (SAFE FILTER)
# ============================================
backends = service.backends(simulator=False, operational=True)

# Filter only devices with enough qubits
valid_backends = [
    b for b in backends
    if b.num_qubits >= N_QUBITS
]

# Sort by queue length (pending jobs)
valid_backends = sorted(valid_backends, key=lambda b: b.status().pending_jobs)

backend = valid_backends[0]

print(f"Using backend: {backend.name}")
print(f"Qubits: {backend.num_qubits} | Pending jobs: {backend.status().pending_jobs}")


# ============================================
# Step 3: Create PennyLane device
# ============================================
dev_ibm = qml.device(
    "qiskit.remote",
    wires=N_QUBITS,
    backend=backend,
    shots=256   # 🔥 reduce for speed
)


# ============================================
# Step 4: Define same VQC
# ============================================
@qml.qnode(dev_ibm)
def vqc_ibm(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))


# ============================================
# Step 5: Prediction (LIMITED RUN)
# ============================================
def q_predict_ibm(X, weights, n_samples=3):
    preds, probs = [], []

    print("\nRunning on IBM Quantum (limited samples)...")

    for i in range(min(n_samples, len(X))):
        raw = vqc_ibm(X[i], weights)
        prob = (raw + 1) / 2
        pred = int(prob > 0.5)

        preds.append(pred)
        probs.append(prob)

        print(f"Sample {i} → Prob: {prob:.3f}, Pred: {pred}")

    return np.array(preds), np.array(probs)


# ============================================
# Step 6: RUN
# ============================================
y_pred_ibm, y_prob_ibm = q_predict_ibm(X_test_q, best_W, n_samples=3)

print("\nTrue labels:", y_test[:len(y_pred_ibm)])
print("Predictions:", y_pred_ibm)

from sklearn.metrics import accuracy_score
acc_ibm = accuracy_score(y_test[:len(y_pred_ibm)], y_pred_ibm)

print("\nIBM Quantum Accuracy (subset):", acc_ibm)

qiskit_runtime_service.__init__:WARNING:2026-04-05 21:57:52,998: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Rakshan. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-05 21:57:53,001: Loading instance: Rakshan, plan: open


Using backend: ibm_fez
Qubits: 156 | Pending jobs: 0

Running on IBM Quantum (limited samples)...
Sample 0 → Prob: 0.425, Pred: 0
Sample 1 → Prob: 0.437, Pred: 0
Sample 2 → Prob: 0.561, Pred: 1

True labels: [0 0 1]
Predictions: [0 0 1]

IBM Quantum Accuracy (subset): 1.0
